# Clinical Text Extraction with Native Ollama

This notebook demonstrates how to extract structured epilepsy-related information from synthetic clinical notes.

The project has two modes:

1. **Rule-based baseline**: simple keyword extraction, no LLM required.
2. **Ollama LLM extraction**: local LLM extraction using native Ollama structured outputs.

All notes are synthetic and are used only for demonstration.

In [5]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from extract_clinical_info import (
    rule_based_extract,
    llm_extract,
    run_extraction,
    ClinicalExtraction,
)

data_path = PROJECT_ROOT / "data" / "synthetic_clinical_notes.csv"
output_path = PROJECT_ROOT / "outputs" / "example_extractions.json"

df = pd.read_csv(data_path)
df.head()

,note_id,clinical_note
0,N001,Synthetic note: Patient with focal epilepsy an...
1,N002,Synthetic note: Patient diagnosed with general...
2,N003,Synthetic note: Patient with suspected tempora...
3,N004,Synthetic note: Episodes were captured on vide...
4,N005,Synthetic note: Patient with focal epilepsy tr...


## 1. Rule-based baseline

This baseline uses simple Python keyword rules. It is included so the project can run without an LLM.

In [6]:
rule_results = run_extraction(
    input_path=data_path,
    output_path=output_path,
    backend="rule-based",
)

pd.DataFrame(rule_results)

Extracting (rule-based): 100%|██████████| 8/8 [00:00<00:00, 3757.07it/s]


,note_id,diagnosis,seizure_type,medications,treatment_response,comorbidities,investigations,clinical_outcome,evidence_quote,confidence
0,N001,focal epilepsy,focal impaired awareness seizures,"[levetiracetam, lamotrigine]",ongoing seizures despite treatment,[anxiety],"[EEG, MRI]",persistent seizures,The patient has tried levetiracetam and lamotr...,high
1,N002,generalised epilepsy,generalised seizures,[valproate],controlled on treatment,[],[EEG],seizure-free or controlled,Seizures are currently controlled on valproate...,high
2,N003,temporal lobe epilepsy,unclear,"[levetiracetam, carbamazepine, lacosamide]",drug-resistant or persistent seizures despite ...,[depression],[MRI],persistent seizures,Seizures persist despite multiple anti-seizure...,high
3,N004,functional non-epileptic seizures,functional non-epileptic seizures,[],unclear,[anxiety],[video-EEG],unclear,Synthetic note: Episodes were captured on vide...,medium
4,N005,focal epilepsy,focal seizures,[lamotrigine],unclear,[migraine],[EEG],reduced seizure frequency,Seizure frequency has reduced from weekly to o...,medium
5,N006,epilepsy,nocturnal seizures,"[levetiracetam, clobazam, topiramate]",ongoing seizures despite treatment,[sleep disturbance],[ECG],persistent seizures,Ongoing nocturnal seizures are reported.,high
6,N007,generalised epilepsy,generalised seizures,"[lamotrigine, ethosuximide]",controlled on treatment,[],[EEG],seizure-free or controlled,No seizures have been reported for two years.,high
7,N008,epilepsy,focal seizures,"[levetiracetam, oxcarbazepine]",drug-resistant or persistent seizures despite ...,[memory complaints],[video-EEG],persistent seizures,The patient has failed trials of oxcarbazepine...,high


## 3. Ollama LLM extraction

Before running this section, make sure Ollama is installed and running.

From your terminal, run:

```bash
ollama pull llama3.1
ollama run llama3.1
```

Then you can run the LLM extraction below.

This may take longer than the rule-based baseline because each note is sent to the local LLM.

In [7]:
llm_results = run_extraction(
     input_path=data_path,
     output_path=output_path,
     backend="llm",
     model="llama3.1",
 )

pd.DataFrame(llm_results)

Extracting (llm): 100%|██████████| 8/8 [11:49<00:00, 88.68s/it]


,note_id,diagnosis,seizure_type,medications,treatment_response,comorbidities,investigations,clinical_outcome,evidence_quote,confidence
0,N001,focal epilepsy,focal impaired awareness seizures,"[levetiracetam, lamotrigine]",ongoing seizures despite treatment,[anxiety],"[EEG, MRI]",persistent seizures,The patient has tried levetiracetam and lamotr...,high
1,N002,generalised epilepsy,generalised seizures,[valproate],controlled on treatment,[],[EEG],seizure-free or controlled,Seizures are currently controlled on valproate...,high
2,N003,temporal lobe epilepsy,focal impaired awareness seizures,"[levetiracetam, carbamazepine, lacosamide]",drug-resistant or persistent seizures despite ...,[depression],[MRI],persistent seizures,Seizures persist despite multiple anti-seizure...,high
3,N004,functional non-epileptic seizures,functional non-epileptic seizures,[],unclear,[anxiety],[video-EEG],unclear,Episodes were captured on video-EEG without ep...,high
4,N005,focal epilepsy,focal seizures,[lamotrigine],controlled on treatment,[migraine],[EEG],reduced seizure frequency,Seizure frequency has reduced from weekly to o...,high
5,N006,epilepsy,nocturnal seizures,"[levetiracetam, clobazam, topiramate]",ongoing seizures despite treatment,[sleep disturbance],[ECG],persistent seizures,Patient with epilepsy following traumatic brai...,high
6,N007,generalised epilepsy,generalised seizures,"[lamotrigine, ethosuximide]",controlled on treatment,[],[EEG],seizure-free or controlled,No seizures have been reported for two years.,high
7,N008,focal epilepsy,focal seizures,"[levetiracetam, oxcarbazepine]",drug-resistant or persistent seizures despite ...,[memory complaints],[video-EEG],persistent seizures,Video-EEG confirmed epileptic seizures.,high


## Notes

- This project uses synthetic data only.
- Do not use real patient data.
- The LLM output is validated with a Pydantic schema.
- The rule-based baseline is included for comparison, not as the main method.